In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 라이브러리

In [ ]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# 시각화
import matplotlib.pyplot as plt

# Tensorflow 라이브러리
import tensorflow as tf

# Tensorflow 전처리
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Tensorflow 모델 구조
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import MultiHeadAttention
from tensorflow.keras.layers import Input, Embedding, Flatten, Multiply, Concatenate, Dropout, Dense, Reshape, Permute
from tensorflow.keras.utils import plot_model

# sklearn 전처리, 데이터 분할, 성능 평가지표
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error


# 데이터셋 로딩

In [ ]:
# 데이터 로딩
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_baseline_emb.pkl')

print(df.shape)
df.head()

In [ ]:
# Encode user and item IDs
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['asin'] = item_encoder.fit_transform(df['asin'])

# 고유한 user와 item 수를 집계
n_users = df['user_id'].nunique()
n_items = df['asin'].nunique()

In [ ]:
n_items

16099

In [ ]:
# train, valid, test으로 데이터셋 분리
train, test = train_test_split(df, test_size=0.2, random_state=42)
train, valid = train_test_split(train, test_size=0.125, random_state=42)

train = train.reset_index(drop=True)
valid = valid.reset_index(drop=True)
test = test.reset_index(drop=True)

In [ ]:
train.shape, valid.shape, test.shape

((46702, 15), (6672, 15), (13344, 15))

In [ ]:
df['rating'].value_counts()

,count
rating,
5,46652
4,14561
3,4002
2,1041
1,462


In [ ]:
# User
user_train = train['user_id'].to_numpy()
user_valid = valid['user_id'].to_numpy()
user_test = test['user_id'].to_numpy()

# Item
item_train = train['asin'].to_numpy()
item_valid = valid['asin'].to_numpy()
item_test = test['asin'].to_numpy()

# rating
y_train = train['rating'].to_numpy()
y_valid = valid['rating'].to_numpy()
y_test = test['rating'].to_numpy()

# NeuMF

In [ ]:
def NeuMF(Num_User, Num_Item, latent_features, predictive_factors, dropout,learning_rate):

    # 입력
    # input layer 설정
    user_input = Input(shape=(1,), dtype='int64')
    item_input = Input(shape=(1,), dtype='int64')

    # GMF 임베딩
    gmf_user_embedding = Embedding(Num_User, latent_features, name='gmf_user_embedding')(user_input)
    gmf_user_embedding = Flatten()(gmf_user_embedding)

    gmf_item_embedding = Embedding(Num_Item, latent_features, name='gmf_item_embedding')(item_input)
    gmf_item_embedding = Flatten()(gmf_item_embedding)

    # MLP 임베딩
    mlp_user_embedding = Embedding(Num_User, latent_features, name='mlp_user_embedding')(user_input)
    mlp_user_embedding = Flatten()(mlp_user_embedding)

    mlp_item_embedding = Embedding(Num_Item, latent_features, name='mlp_item_embedding')(item_input)
    mlp_item_embedding = Flatten()(mlp_item_embedding)

    # GMF 레이어
    gmf_mul = Multiply()([gmf_user_embedding, gmf_item_embedding])

    # MLP 레이어
    mlp_concat = Concatenate()([mlp_user_embedding, mlp_item_embedding])
    mlp_dropout = Dropout(dropout)(mlp_concat)

    # 예측 레이어에 따라 MLP 레이어 조정
    if predictive_factors == 8:
        mlp_layer_sizes = [32, 16, 8]
    elif predictive_factors == 16:
        mlp_layer_sizes = [64, 32, 16]
    elif predictive_factors == 32:
        mlp_layer_sizes = [128, 64, 32]
    elif predictive_factors == 64:
        mlp_layer_sizes = [256, 128, 64]
    else:
        raise ValueError("Unsupported predictive factors size.")

    for idx, size in enumerate(mlp_layer_sizes):
        mlp_dropout = Dense(units=size, activation='relu', name=f'mlp_layer{idx+1}')(mlp_dropout)
        if idx < len(mlp_layer_sizes) - 1:  # Add dropout except for the last layer
            mlp_dropout = Dropout(0.1)(mlp_dropout)

    # GMF와 MLP 결합
    merged_vector = tf.keras.layers.concatenate([gmf_mul, mlp_dropout])

    # 출력
    output_layer = Dense(1, name='output_layer')(merged_vector)

    # 모델 컴파일
    model = Model(inputs = [user_input, item_input], outputs = output_layer)
    model.compile(optimizer = tf.keras.optimizers.SGD(learning_rate = learning_rate), loss = "mse", metrics = ["mse", "mae"])

    return model


In [ ]:
# 파라미터 설정
Num_User = n_users + 1
Num_Item = n_items + 1
latent_features = 16
predictive_factors = 8
dropout = 0.2
learning_rate = 0.0001

In [ ]:
# Configuration of EarlyStopping
early_stopping = EarlyStopping(
    monitor ='val_loss',  # Monitor model's validation loss
    min_delta = 0.001,     # Minimum change to qualify as an improvement
    patience = 5,         # Stop after 5 epochs without improvement
    verbose = 1,           # Print messages
    mode = 'min',          # Stop training when the quantity monitored has stopped decreasing
    restore_best_weights = True  # Restore model weights from the epoch with the best value of the monitored quantity
)

In [ ]:
# 모델의 구성 확인
model = NeuMF(Num_User, Num_Item, latent_features, predictive_factors, dropout,learning_rate)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_user_embedding  │ (None, 1, 16)     │    119,200 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_item_embedding  │ (None, 1, 16)     │    257,600 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 16)        │          0 │ mlp_user_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 16)        │          0 │ mlp_item_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 32)        │          0 │ flatten_2[0][0],  │
│ (Concatenate)       │                   │            │ flatten_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 32)        │          0 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_layer1 (Dense)  │ (None, 32)        │      1,056 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ mlp_layer1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gmf_user_embedding  │ (None, 1, 16)     │    119,200 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gmf_item_embedding  │ (None, 1, 16)     │    257,600 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_layer2 (Dense)  │ (None, 16)        │        528 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 16)        │          0 │ gmf_user_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 16)        │          0 │ gmf_item_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 16)        │          0 │ mlp_layer2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 16)        │          0 │ flatten[0][0],    │
│                     │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_layer3 (Dense)  │ (None, 8)         │        136 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 24)        │          0 │ multiply[0][0],   │
│ (Concatenate)       │                   │            │ mlp_layer3[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_layer        │ (None, 1)         │         25 │ concatenate_1[0]… │
│ (Dense)             │                   │            │                 

 Total params: 755,345 (2.88 MB)

 Trainable params: 755,345 (2.88 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 모델 학습
history = model.fit([user_train, item_train], y_train,
                    validation_data=([user_valid, item_valid], y_valid),
                    epochs = 5000,
                    batch_size = 128,
                    callbacks=[early_stopping])

In [ ]:
# 모델 성능 측정
predicted_ratings = model.predict([user_test, item_test])

# MAE
MAE = mean_absolute_error(y_test, predicted_ratings)
print(f"MAE: {MAE:.4f}")

# MSE
MSE = mean_squared_error(y_test, predicted_ratings)
print(f"MSE: {MSE:.4f}")

# RMSE
RMSE = np.sqrt(MSE)
print(f"RMSE: {RMSE:.4f}")

# MAPE
MAPE = mean_absolute_percentage_error(y_test, predicted_ratings) * 100
print(f"MAPE: {MAPE:.4f}%")

417/417 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
MAE: 0.5769
MSE: 0.5357
RMSE: 0.7319
MAPE: 16.6889%


In [4]:
import pandas as pd

# 주어진 값들
data = {
    "MAE": [0.4503, 0.4525, 0.4315, 0.4274, 0.4438],
    "MSE": [0.3980, 0.3883, 0.3863, 0.3796, 0.3980],
    "RMSE": [0.6309, 0.6232, 0.6215, 0.6161, 0.6308],
    "MAPE": [0.1301, 0.1255, 0.1270, 0.1276, 0.1302],
}

# DataFrame 생성
df = pd.DataFrame(data)

# 평균과 표준편차 계산
results = df.agg(['mean', 'std']).T

# 결과 출력
print(results)

         mean       std
MAE   0.44110  0.011200
MSE   0.39004  0.007949
RMSE  0.62450  0.006362
MAPE  0.12808  0.002039
